In [16]:
! wb resource resolve --id=workspace-bucket

gs://workspace-bucket-wb-radiant-cabbage-3726


In [10]:
%%bash 

WORKSPACE_BUCKET=$(wb resource resolve --id=workspace-bucket)

cat > /home/jupyter/nextflow.config <<EOF
profiles {
  gcb {
      process.executor = "google-batch"
      process.container = "florianzink/nf-gwas-gcloud:v0.3"
      workDir = "$WORKSPACE_BUCKET/workflows/nextflow-scratch"

      google.location = "us-central1"
      google.project = "$GOOGLE_CLOUD_PROJECT"
      google.enableRequesterPaysBuckets = true
      google.batch.debug = true
      google.batch.serviceAccountEmail = "${PET_SA_EMAIL}"
      google.batch.network = "global/networks/network"
      google.batch.subnetwork = "regions/us-central1/subnetworks/subnetwork"
      google.batch.usePrivateAddress = true
      google.batch.copyImage = "gcr.io/google.com/cloudsdktool/cloud-sdk:alpine"
      google.batch.bootDiskSize = "50.GB"
  }
}
EOF


In [18]:
%%file /home/jupyter/exome_specific.nf

WORKSPACE_BUCKET="gs://workspace-bucket-wb-radiant-cabbage-3726"

nextflow.enable.dsl=2

params.genome_bgens = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/bgen"
params.exome_bgens = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/exome/bgen"
params.exome_only_bgens = "${WORKSPACE_BUCKET}/exome_only"


process MakeExomeOnly {
    tag { chr }
    container "florianzink/nf-gwas-gcloud:v0.3"
    scratch true
    disk '50GB'
    cpus 4
    memory '8 GB'
   	
    publishDir "${params.exome_only_bgens}"

    input:
    tuple val(chr), path(genome_bgi, stageAs: 'genome.bgi'), path(exome_bgen), path(exome_bgi)

    output:
    path("chr${chr}_exome_only.bgen")
    path("chr${chr}_exome_only.bgen.bgi")

    script:
    """
    # 1. extract variant_ids from sqlite indices
    sqlite3 ${genome_bgi} "SELECT rsid FROM Variant;" | sort -u > genome.ids
    sqlite3 ${exome_bgi}  "SELECT rsid FROM Variant;" | sort -u > exome.ids

    # 2. compute exome-only ids
    comm -23 exome.ids genome.ids > exome_only.ids

    # 4. loop over chunks and extract variants
    out_bgen=chr${chr}_exome_only.bgen
    bgenix -g ${exome_bgen} -incl-rsids exome_only.ids > \$out_bgen
    bgenix -g \$out_bgen -index
    """
}



workflow {
    Channel
        .fromPath("${params.genome_bgens}/acaf_threshold.chr*.bgen.bgi")
        .map { genome_bgi ->
            def chr = genome_bgi.name.replaceFirst(/acaf_threshold.chr([0-9XYMT]+).bgen.bgi/, "\$1")
            tuple(
                chr,
                file(genome_bgi),           // unique name
                file("${params.exome_bgens}/exome.chr${chr}.bgen"),
                file("${params.exome_bgens}/exome.chr${chr}.bgen.bgi")
            )
        }
        | MakeExomeOnly
}



Overwriting /home/jupyter/exome_specific.nf


```
cd /home/jupyter
nextflow run exome_specific.nf -profile gcb
```
